In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentence = "Tech companies laid off thousands of employees in 2023"
embedding = model.encode(sentence)

print("Embedding shape:", embedding.shape)
print("First 10 values:", embedding[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (384,)
First 10 values: [-0.01283925 -0.01212105  0.06105979 -0.00437263 -0.05858351  0.01646333
 -0.01595696 -0.01153123 -0.04041953 -0.02047331]


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

sentences = [
    "Tech companies laid off thousands of employees in 2023",
    "Many workers lost their jobs in the technology sector",
    "I love eating pizza on weekends",
    "AI is transforming how businesses automate decisions"
]

embeddings = model.encode(sentences)

# Compare first sentence to all others using cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

for i in range(1, len(sentences)):
    sim = cosine_similarity(embeddings[0], embeddings[i])
    print(f"Similarity with: '{sentences[i]}'")
    print(f"Score: {sim:.4f}\n")

Similarity with: 'Many workers lost their jobs in the technology sector'
Score: 0.5863

Similarity with: 'I love eating pizza on weekends'
Score: -0.0266

Similarity with: 'AI is transforming how businesses automate decisions'
Score: 0.2081



In [2]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="news_test")

documents = [
    "Tech companies laid off thousands of employees in 2023",
    "Many workers lost their jobs in the technology sector",
    "I love eating pizza on weekends",
    "AI is transforming how businesses automate decisions",
    "Oracle announced major layoffs affecting India operations",
    "The stock market crashed due to inflation concerns"
]

collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print("Documents added:", collection.count())

/Users/sashipraneethmuthyala/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.ta


Documents added: 6


In [3]:
results = collection.query(
    query_texts=["companies firing workers"],
    n_results=3
)

for i, doc in enumerate(results['documents'][0]):
    print(f"{i+1}. {doc}")
    print(f"   Distance: {results['distances'][0][i]:.4f}\n")

1. Tech companies laid off thousands of employees in 2023
   Distance: 1.1532

2. Many workers lost their jobs in the technology sector
   Distance: 1.2399

3. Oracle announced major layoffs affecting India operations
   Distance: 1.3820



In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client_llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

question = "Which companies had layoffs?"

# Step 1 - Retrieve relevant context
results = collection.query(query_texts=[question], n_results=3)
context = "\n".join(results['documents'][0])

# Step 2 - Pass context + question to LLM
response = client_llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer the question using only the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
)

print(response.choices[0].message.content)

Tech companies, including Oracle, had layoffs affecting their operations, particularly in the technology sector.
